# 实验二 · 矩阵向量乘法：数据分解与线程传参

**所属**：《并行计算》第四章 · Pthread 多线程编程　|　**难度**：⭐⭐ 基础　|　**预计时长**：20–30 分钟

> **实验说明**
> 1. 实验一中，线程只接收一个整数序号。真实的并行任务需要向线程传递多个参数。本实验以矩阵向量乘法（GEMV）为载体，对比两种传参方案——**全局变量**与**参数结构体**，并说明工程代码应当选择后者的理由。
> 2. 本实验同时给出本章第一个不需要任何同步机制的并行案例，用以建立一条重要判据：**并非所有共享数据都需要加锁**。
> 3. 请自上而下依次执行各单元格（Shift+Enter）。
> 4. 本实验涉及性能测量。请尽量在**华为鲲鹏多核处理器**上运行；单核环境下加速比恒约为 1，无法体现并行收益。
> 5. 遇到 🔧 **动手练习** 与 🤔 **思考题** 时，建议先独立完成，再阅读后续内容。

## 🎯 学习目标

完成本实验后，学生应能够：

- 实现按行划分的**数据分解**，并正确处理行数不能被线程数整除的情形
- 对比**全局变量传参**与**参数结构体传参**两种方案，说明各自的适用边界
- 运用数据竞争的严格定义，论证本实验的并行版本为何无需任何锁
- 指出「所有线程共用同一个参数对象」这一典型错误，并说明其后果
- 使用统一的计时方法测量加速比，并结合**计算访存比**解释实测结果
- 说明性能对比实验的两项前提：计算代码一致、核心类型同构
- 定量测量线程创建与销毁的开销，判断其在何种条件下不可忽略

## 🗺️ 学习路径

1. **准备阶段**：理解 GEMV 的算法结构，明确按行块划分为何天然适合并行
2. **判据建立**：分析 `A`、`x`、`y` 三者的访问模式，论证并行版本无需同步
3. **版本一**：用全局变量向线程传递数据，运行并测量
   → 分析该方案在函数复用性与可维护性上的局限
4. **版本二**：用参数结构体向线程传递数据，运行并测量
   → 掌握「每线程一个独立参数对象」这一必要条件
5. **对比与扩展**：比较两版的性能与工程质量，并测量加速比随线程数的变化
   → 结合访存受限特征解释加速比为何低于线程数
6. **开销测量**：单独测量线程创建与销毁的代价，判断其适用边界
   → 引出线程池思想，为实验四、五做准备

## 1. 背景与动机

实验一的线程只打印一行文本，不涉及任何数据。本实验进入真实的计算任务：**矩阵向量乘法**（General Matrix-Vector multiplication, GEMV）。

选择这一算法作为第二个案例，有三方面考虑：

- **结构简单且可验证**。算法本身只是一个双重循环，结果可与串行版本逐元素比对，便于判定并行实现是否正确。
- **具备真实的并行价值**。不同于 Hello World，GEMV 有实际的计算量，可以测量加速比。
- **恰好不需要同步**。它是本章唯一一个「多线程写同一个数组却无需加锁」的案例，正适合用来建立判断同步必要性的准则。

GEMV 属于 BLAS Level-2 例程，是神经网络全连接层、迭代法求解线性方程组等场景的核心计算。第三章曾用 ARM NEON 优化过类似的向量运算，本实验则从**线程级并行**的角度处理同一类问题。

## 2. 算法与数据分解

矩阵向量乘法 $y = Ax$，其中 $A$ 为 $m \times n$ 矩阵，$x$ 为长度 $n$ 的向量，$y$ 为长度 $m$ 的向量：

$$y_i = \sum_{j=0}^{n-1} A_{ij} \cdot x_j , \qquad i = 0, 1, \dots, m-1$$

串行实现是一个标准的双重循环：

```c
for (long i = 0; i < rows; ++i) {
  float sum = 0.0f;
  for (long j = 0; j < cols; ++j) {
    sum += A[i * cols + j] * x[j];
  }
  y[i] = sum;
}
```

### 2.1 该问题为何适合并行

观察外层循环：**第 $i$ 次迭代只写 `y[i]`，且不读取 `y` 的任何元素**。这意味着不同的 $i$ 之间没有数据依赖，可以任意顺序执行，也可以同时执行。这类循环称为**可并行循环**（parallelizable loop），是最容易并行化的一类。

### 2.2 按行块划分

把 $m$ 行平均分给 $t$ 个线程，线程 $r$ 负责行区间 $[\text{first}_r,\ \text{last}_r)$：

```c
long local_rows   = rows / thread_count;
long my_first_row = my_rank * local_rows;
long my_last_row  = (my_rank == thread_count - 1) ? rows
                                                  : my_first_row + local_rows;
```

最后一个线程负责到 `rows` 为止，从而吸收 `rows % thread_count` 的余数。这样写不会遗漏任何一行，代价是最后一个线程最多多承担 `thread_count - 1` 行；在行数远大于线程数时，这一负载差异可以忽略。

### 2.3 💡 关键判据：本实验为何不需要任何锁

`A`、`x`、`y` 都是所有线程可见的共享数据，但本实验一把锁都没有使用。逐一考察三者的访问模式：

<!--
| 数据 | 访问模式 | 是否构成数据竞争 |
|---|---|---|
| `A` | 所有线程**只读** | 否——数据竞争要求至少有一个写者 |
| `x` | 所有线程**只读** | 否——同上 |
| `y` | 被写，但**每个线程写互不重叠的区间** | 否——不同线程访问的不是同一个内存位置 |
-->
<table>
  <thead>
    <tr>
      <th style="text-align: left;">数据</th>
      <th style="text-align: left;">访问模式</th>
      <th style="text-align: left;">是否构成数据竞争</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><code>A</code></td>
      <td style="text-align: left;">所有线程<strong>只读</strong></td>
      <td style="text-align: left;">否——数据竞争要求至少有一个写者</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>x</code></td>
      <td style="text-align: left;">所有线程<strong>只读</strong></td>
      <td style="text-align: left;">否——同上</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>y</code></td>
      <td style="text-align: left;">被写，但<strong>每个线程写互不重叠的区间</strong></td>
      <td style="text-align: left;">否——不同线程访问的不是同一个内存位置</td>
    </tr>
  </tbody>
</table>

线程 0 写 `y[0..k)`，线程 1 写 `y[k..2k)`，二者永远不会写到同一个元素。由此得到一条贯穿全章的判据：

> 需要同步的不是「共享数据」，而是「**被并发访问的同一个内存位置，且其中至少有一个是写操作**」。
> 只读共享、以及写入互不重叠的区间，都不需要任何同步机制。

牢固掌握这一判据，可以避免大量不必要的加锁——而不必要的锁正是并行程序性能低下的常见原因（详见实验三）。

## 3. 核心问题：如何向线程传递多个参数

回顾 `pthread_create` 的函数原型：

```c
int pthread_create(pthread_t *thread, const pthread_attr_t *attr,
                   void *(*start_routine)(void *),
                   void *arg);            // 只有一个参数
```

线程入口函数**只能接收一个 `void *` 参数**。实验一中需要传递的仅是一个整数序号，故可将其按值转换为 `void *`。但本实验的线程需要知道 `A`、`x`、`y`、`rows`、`cols`、`thread_count` 以及自身的 `rank`——共七项数据。

标准的解决途径有两条，本实验各实现一次并加以对比：

<!--
| 方案 | 做法 | 本实验对应版本 |
|---|---|---|
| **全局变量** | 数据置于全局作用域，线程函数只接收 `rank`，其余从全局读取 | 版本一 |
| **参数结构体** | 将全部数据打包为结构体，传递其地址 | 版本二 |
-->
<table>
  <thead>
    <tr>
      <th style="text-align: left;">方案</th>
      <th style="text-align: left;">做法</th>
      <th style="text-align: left;">本实验对应版本</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><strong>全局变量</strong></td>
      <td style="text-align: left;">数据置于全局作用域，线程函数只接收 <code>rank</code>，其余从全局读取</td>
      <td style="text-align: left;">版本一</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>参数结构体</strong></td>
      <td style="text-align: left;">将全部数据打包为结构体，传递其地址</td>
      <td style="text-align: left;">版本二</td>
    </tr>
  </tbody>
</table>

### ⚠️ 一个必须避免的错误

无论采用哪种方案，都不能让所有线程共用同一个可变的参数对象。以下写法是错误的：

```c
for (long i = 0; i < thread_count; ++i) {
  pthread_create(&handles[i], NULL, MatVec_thread, (void *)&i);   // 错误
}
```

所有线程接收到的是**同一个变量 `i` 的地址**。主线程仍在循环中递增 `i`，线程读到的值完全取决于其被调度的时刻；且循环结束后 `i` 可能已经失效。这与实验一中「不能传 `&thread`」是同一个错误。

正确的做法只有两种：**按值传递**（实验一），或**为每个线程准备独立的参数对象**（本实验版本二）。

## 4. 环境准备

下面的单元格检查编译器与硬件环境，并定义本实验统一使用的编译、运行、结果解析与绘图工具函数。

**请留意 CPU 核心数**：加速比不可能超过物理核心数。核心数为 1 时，并行版本与串行版本耗时基本相同，无法体现并行收益。

In [ ]:
import platform, subprocess, shutil, sys, os, re

print("Python  :", sys.version.split()[0])
print("架构    :", platform.machine())
CC = shutil.which("gcc") or shutil.which("clang") or shutil.which("cc")
print("编译器  :", CC)
NCPU = os.cpu_count()
print("CPU 核心:", NCPU)

if CC is None:
    print(
        "\n⚠️  未找到 C 编译器，请先安装 gcc（如 sudo apt install build-essential）。"
    )
elif NCPU == 1:
    print("\n⚠️  当前仅 1 个核心：加速比恒约为 1，无法体现并行收益，")
    print("    建议在华为鲲鹏多核处理器上运行本实验。")
else:
    print(f"\n✅ 环境就绪：编译器可用，{NCPU} 核可用，可以开始实验！")


### 核心同构性检查

性能对比还有一个前提：**参与比较的各次测量必须运行在同类型的核心上**。近年的处理器普遍采用**大小核**架构（ARM big.LITTLE、Intel P-core/E-core），同一颗 CPU 上的核心性能可以相差一倍以上。若串行基准恰好运行在大核、而工作线程被调度到小核，测得的加速比将完全失真。

下面的单元格通过各核心的最高频率来识别核心类型。

In [ ]:
import glob


def detect_core_types():
    """按最高频率给核心分组；返回 {频率(kHz): [核心编号]}，无信息时返回 None。"""
    freqs = {}
    for path in sorted(
        glob.glob("/sys/devices/system/cpu/cpu[0-9]*/cpufreq/cpuinfo_max_freq")
    ):
        cpu = int(path.split("/")[5][3:])
        try:
            freqs[cpu] = int(open(path).read().strip())
        except OSError:
            pass
    if not freqs:
        return None
    groups = {}
    for cpu, f in freqs.items():
        groups.setdefault(f, []).append(cpu)
    return dict(sorted(groups.items(), reverse=True))


def cpu_list(cpus):
    """把核心编号压缩为 taskset 可用的区间表示，如 0-7,12."""
    cpus = sorted(cpus)
    parts, s, prev = [], cpus[0], cpus[0]
    for c in cpus[1:] + [None]:
        if c is not None and c == prev + 1:
            prev = c
            continue
        parts.append(str(s) if s == prev else f"{s}-{prev}")
        s = prev = c
    return ",".join(parts)


groups = detect_core_types()
if groups is None:
    print("未读取到 cpufreq 信息（容器或虚拟机中常见），无法判断核心类型。")
    print("若宿主机为大小核处理器，请参考下方说明使用 taskset 绑定核心后再测量。")
elif len(groups) == 1:
    print(
        f"✅ 全部 {len(next(iter(groups.values())))} 个核心同构，可直接进行性能对比。"
    )
else:
    print(f"⚠️  检测到 {len(groups)} 种不同类型的核心（大小核架构）：")
    for f, cpus in groups.items():
        print(f"    {f/1e6:.2f} GHz : {len(cpus):2d} 个核心  ->  {cpu_list(cpus)}")
    big = cpu_list(next(iter(groups.values())))
    print("\n直接测量所得的加速比不可信。请绑定到同一类核心后重新测量，例如：")
    print(f"    taskset -c {big} ./src_gemv/pthread_mat_vec_struct 4000 4000 4")

### 编译、运行与绘图工具函数

全章统一的编译选项为：

```bash
gcc -O3 -fPIC -pthread <source>.c -o <target> -lpthread -lm
```

- `-O3`：开启完整优化。性能测量须在与生产环境一致的优化级别下进行，否则所得加速比没有参考价值。
- `-pthread` / `-lpthread`：链接 POSIX 线程库。
- `-lm`：本实验的校验函数使用 `fabs`，需链接数学库。
- **不使用 `-march=native`**：该选项会针对当前机器生成专用指令，使各版本对比失去公平性，且生成的二进制无法在其他机型上运行。

In [3]:
SRC_DIR = "src_gemv"  # 源代码目录
os.makedirs(SRC_DIR, exist_ok=True)


def compile_c(src, out):
    """用全章统一选项编译一个源文件，成功返回可执行文件名，失败返回 None。"""
    base = shutil.which("gcc") or shutil.which("cc") or "cc"
    cmd = f"{base} -O3 -fPIC -pthread -Wall -Wextra {src} -o {out} -lpthread -lm"
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode == 0:
        print("✅ 编译成功：", cmd)
        if r.stderr.strip():
            print(r.stderr.strip())
        return out
    print("❌ 编译失败：\n", r.stderr)
    return None


def run_bin(out, *args, echo=True):
    """运行可执行文件并返回其标准输出；echo=True 时同时打印。"""
    r = subprocess.run(
        ["./" + out] + [str(a) for a in args], capture_output=True, text=True
    )
    if echo:
        print(r.stdout, end="")
        if r.returncode != 0 and r.stderr:
            print("STDERR:", r.stderr)
    return r.stdout


def parse_result(text):
    """解析程序输出，返回 {serial, parallel, speedup, check}。"""

    def grab(pat):
        m = re.search(pat, text)
        return float(m.group(1)) if m else None

    chk = re.search(r"Check\s*:\s*(\w+)", text)
    return {
        "serial": grab(r"Serial time\s*:\s*([\d.]+)"),
        "parallel": grab(r"Parallel time\s*:\s*([\d.]+)"),
        "speedup": grab(r"Speedup\s*:\s*([\d.]+)"),
        "check": chk.group(1) if chk else None,
    }


In [4]:
import matplotlib.pyplot as plt

def plot_speedup(labels, speedups, title="", xlabel=""):
    """绘制加速比柱状图：灰=无收益，蓝=有收益，红=最优；虚线为串行基准。"""
    best = speedups.index(max(speedups))
    colors = ["#9aa0a6" if s <= 1.05 else "#295E96" for s in speedups]
    colors[best] = "#C7000B"
    fig, ax = plt.subplots(figsize=(8, 4.2))
    bars = ax.bar([str(l) for l in labels], speedups, color=colors)
    ax.axhline(1.0, ls="--", c="gray", lw=1.2, label="Serial baseline")
    for b, s in zip(bars, speedups):
        ax.text(b.get_x() + b.get_width() / 2, s, f"{s:.2f}x",
                ha="center", va="bottom", fontsize=10)
    ax.set_ylabel("Speedup")
    if xlabel:
        ax.set_xlabel(xlabel)
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()

## 5. 版本一：全局变量传参

第一种方案把线程所需的全部数据置于全局作用域，线程函数只接收 `rank`：

```c
long rows = 0;
long cols = 0;
int thread_count = 0;
float *A = NULL;
float *x = NULL;
float *y = NULL;

void *MatVec_thread(void *rank) {
  long my_rank = (long)rank;
  ...                          // rows、cols、A、x、y 均从全局作用域取得
}
```

### 代码要点

- **线程函数签名极简**：只需传递一个整数序号，沿用实验一的按值传递技巧。
- **计时方法**：串行与并行版本各重复 `NTIMES = 20` 次，取**平均值**而非最优值。取平均能反映真实的期望性能；取最优值会系统性地低估调度开销与缓存抖动的影响。
- **共用同一个计算内核**：串行基准与工作线程都调用 `MatVec_rows`，而不是各自写一遍循环。其原因见下方的方法论说明。
- **正确性校验**：`check_diff` 逐元素比较并行结果与串行结果，容差为相对误差 `1e-5`。浮点加法不满足结合律，故不能要求逐位相等。

### ⚠️ 方法论：串行基准与并行版本必须执行同一段代码

性能对比实验有一条容易被忽视的前提：**被比较的两个版本，其计算部分必须编译成等价的机器码**。否则测得的差异并非来自并行，而是来自编译器对两段代码的不同处理。

一种常见的错误写法是把同一个计算循环**书写两遍**——一遍放在串行函数里，一遍放在线程函数里：

```c
static void MatVec_serial(...) {       // 第一遍：static，且只有一处调用
  for (...) { ... }                    // 编译器会将其内联进 main，掌握完整的别名信息
}

void *MatVec_thread(void *arg) {       // 第二遍：线程入口，非 static
  for (...) { ... }                    // 编译器必须保守处理，优化程度可能不同
}
```

两段循环源码相同，但**所处的优化上下文不同**：前者被内联后，编译器能确定 `A`、`x`、`y` 来自各自独立的分配、互不重叠；后者的指针经由参数传入，编译器无法作同样的判断。二者的向量化决策因此可能分歧。

一旦分歧，1 个线程的并行版本就会与串行版本产生明显的耗时差异——而这两者做的其实是完全相同的工作。此时测得的所有加速比都失去意义。

**本实验的处理方式**：把计算逻辑抽出为唯一的内核函数 `MatVec_rows`，串行基准与工作线程都调用它。

```c
static void MatVec_rows(const float *A_in, const float *x_in, float *y_out,
                        long first_row, long last_row, long n_cols);

MatVec_rows(A, x, y_serial, 0, rows, cols);                      // 串行基准
MatVec_rows(A, x, y, my_first_row, my_last_row, cols);           // 工作线程
```

如此一来，两条路径的计算部分来自同一份源码，编译器的优化决策必然一致，测得的差异才真正归因于并行本身。

> **自检方法**：将线程数设为 **1** 运行。此时并行版本与串行版本的工作量完全相同，二者耗时应当接近（差距仅为一次线程创建，约数十微秒）。**若 1 线程的加速比明显偏离 1.0，说明基准本身存在问题，此时所有加速比数据都不可信**，须先排查原因再继续。

可用下列命令查看编译器对各循环的向量化决策，确认二者来自同一行源码：

```bash
gcc -O3 -fPIC -pthread -fopt-info-vec src_gemv/pthread_mat_vec_struct.c \
    -o /dev/null -lpthread -lm
```

In [ ]:
%%writefile {SRC_DIR}/pthread_mat_vec_global.c
#include <math.h>
#include <pthread.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

#define MAX_THREADS 64
#define NTIMES 20

// Every thread reads these globals. The thread function takes only its rank
// and recovers everything else from the global scope. This works, but it
// couples the thread function to one specific set of global objects.
long rows = 0;
long cols = 0;
int thread_count = 0;
float* A = NULL;
float* x = NULL;
float* y = NULL;

static double get_time_ms(void) {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

// Returns "PASS" when every element agrees within the tolerance.
static const char* check_diff(const float* ref, const float* test, long n) {
  for (long i = 0; i < n; ++i) {
    if (fabs(ref[i] - test[i]) > 1e-5 * fabs(ref[i]) + 1e-5) {
      return "FAIL";
    }
  }
  return "PASS";
}

// Computes rows [first_row, last_row) of y = A*x. The serial baseline and the
// worker threads both call this function, so the two timings compare the same
// machine code instead of two separately compiled copies of the same loop.
static void MatVec_rows(const float* A_in, const float* x_in, float* y_out,
                        long first_row, long last_row, long n_cols) {
  for (long i = first_row; i < last_row; ++i) {
    float sum = 0.0f;
    for (long j = 0; j < n_cols; ++j) {
      sum += A_in[i * n_cols + j] * x_in[j];
    }
    y_out[i] = sum;
  }
}

// Thread entry function. The only argument is the rank; all data comes from
// globals.
void* MatVec_thread(void* rank) {
  long my_rank = (long)rank;

  // Block partitioning over rows. The last thread absorbs the remainder.
  long local_rows = rows / thread_count;
  long my_first_row = my_rank * local_rows;
  long my_last_row =
      (my_rank == thread_count - 1) ? rows : my_first_row + local_rows;

  // Each thread writes a disjoint slice of y, so no synchronization is needed.
  // Repeat inside the thread so the workers are created once instead of
  // NTIMES times. Short-lived threads are placed poorly by the scheduler on
  // heterogeneous (big.LITTLE) CPUs, which distorts the measurement.
  for (int t = 0; t < NTIMES; ++t) {
    MatVec_rows(A, x, y, my_first_row, my_last_row, cols);
  }

  return NULL;
}

int main(int argc, char* argv[]) {
  if (argc != 4) {
    fprintf(stderr, "Usage: %s <rows> <cols> <thread_count>\n", argv[0]);
    return 1;
  }

  rows = strtol(argv[1], NULL, 10);
  cols = strtol(argv[2], NULL, 10);
  thread_count = (int)strtol(argv[3], NULL, 10);

  if (rows <= 0 || cols <= 0) {
    fprintf(stderr, "Error: rows and cols must be positive\n");
    return 1;
  }
  if (thread_count <= 0 || thread_count > MAX_THREADS) {
    fprintf(stderr, "Error: thread_count must be between 1 and %d\n",
            MAX_THREADS);
    return 1;
  }

  A = malloc(rows * cols * sizeof(float));
  x = malloc(cols * sizeof(float));
  y = malloc(rows * sizeof(float));
  float* y_serial = malloc(rows * sizeof(float));
  pthread_t* thread_handles = malloc(thread_count * sizeof(pthread_t));

  if (A == NULL || x == NULL || y == NULL || y_serial == NULL ||
      thread_handles == NULL) {
    fprintf(stderr, "Error: memory allocation failed\n");
    return 1;
  }

  for (long i = 0; i < rows * cols; ++i) A[i] = (float)(i % 100) * 0.01f;
  for (long j = 0; j < cols; ++j) x[j] = (float)(j % 50) * 0.02f;

  printf("Matrix-Vector Multiply (global variables)\n");
  printf("Size: %ld x %ld, Threads: %d, Repeats: %d\n\n", rows, cols,
         thread_count, NTIMES);

  // Serial baseline. Report the average, not the best, of NTIMES runs.
  double start = get_time_ms();
  for (int t = 0; t < NTIMES; ++t) {
    MatVec_rows(A, x, y_serial, 0, rows, cols);
  }
  double time_serial = (get_time_ms() - start) / NTIMES;

  // Parallel version.
  start = get_time_ms();
  for (long i = 0; i < thread_count; ++i) {
    int rc = pthread_create(&thread_handles[i], NULL, MatVec_thread, (void*)i);
    if (rc != 0) {
      fprintf(stderr, "Error: pthread_create failed\n");
      return 1;
    }
  }
  for (long i = 0; i < thread_count; ++i) {
    pthread_join(thread_handles[i], NULL);
  }
  double time_parallel = (get_time_ms() - start) / NTIMES;

  printf("Serial time   : %8.3f ms\n", time_serial);
  printf("Parallel time : %8.3f ms\n", time_parallel);
  printf("Speedup       : %8.2fx\n", time_serial / time_parallel);
  printf("Check         : %s\n", check_diff(y_serial, y, rows));

  free(A);
  free(x);
  free(y);
  free(y_serial);
  free(thread_handles);
  return 0;
}

编译并运行版本一。

In [ ]:
gemv_global = compile_c(f"{SRC_DIR}/pthread_mat_vec_global.c",
                        f"{SRC_DIR}/pthread_mat_vec_global")
print()
NT = max(2, min(4, os.cpu_count()))      # 线程数：不超过 4，且至少为 2
out_global = run_bin(gemv_global, 4000, 4000, NT)

### 💡 版本一的局限

版本一可以正确工作，写法也最为直接。但从工程角度看，它存在四项局限：

1. **函数不可复用**。`MatVec_thread` 与那一组特定的全局变量绑定。若需在同一程序中对两个不同的矩阵同时做乘法，该函数无法胜任。
2. **不可重入**。函数行为依赖外部状态，同一时刻只能存在一组有效参数。
3. **命名空间污染**。`A`、`x`、`y`、`rows`、`cols` 这些标识符占据了整个文件的全局作用域，在大型项目中极易与其他模块冲突。
4. **不利于推理**。阅读函数时无法从签名判断它读写了哪些数据，必须通读函数体。这一点在排查并发缺陷时代价高昂——而并发缺陷本就最难排查。

上述局限在数十行的教学程序中并不明显，但在工程代码中会显著影响可维护性。

## 6. 版本二：参数结构体传参

第二种方案把线程所需的一切打包进结构体，`pthread_create` 传递结构体的地址：

```c
typedef struct {
  long my_rank;
  long rows;
  long cols;
  int thread_count;
  const float *A;
  const float *x;
  float *y;
} ThreadData;
```

### 6.1 ⚠️ 必须使用数组而非单个变量

正确的做法是为每个线程分配**独立的**参数对象：

```c
ThreadData *thread_data = malloc(thread_count * sizeof(ThreadData));
for (long i = 0; i < thread_count; ++i) {
  thread_data[i].my_rank = i;              // 每个线程拥有自己的参数对象
  ...
  pthread_create(&thread_handles[i], NULL, MatVec_thread, &thread_data[i]);
}
```

若写成下面这样，则会重蹈第 3 节所述的覆辙：

```c
ThreadData data;                           // 错误：全部线程共用一份
for (long i = 0; i < thread_count; ++i) {
  data.my_rank = i;                        // 主线程持续改写同一个对象
  pthread_create(&handles[i], NULL, MatVec_thread, &data);
}
```

所有线程获得的是同一个地址，而主线程仍在循环中改写其内容。线程读到的 `my_rank` 完全取决于其被调度的时刻——这是一处确实存在的数据竞争。

### 6.2 参数对象的生命周期

参数对象的生命周期**必须覆盖线程的整个生命周期**。本实验用 `malloc` 分配、在 `main` 末尾 `free`，满足这一要求。

若将 `ThreadData data[N]` 声明为某个函数的局部变量，而该函数在线程结束之前就返回，则线程将访问已经失效的栈内存，属于未定义行为。这与实验一中「按值传递不存在生命周期问题」形成对照：**一旦改为传址，就必须对生命周期负责**。

### 6.3 `const` 的作用

`A` 与 `x` 声明为 `const float *`，在类型层面表达了「本线程只读这两块数据」的意图。这并非修饰性写法：

- 它使编译器能在无意写入时报错；
- 它让阅读代码的人一眼看出哪些数据是只读共享的，因而**不需要同步**（参见第 2.3 节的判据）。

In [ ]:
%%writefile {SRC_DIR}/pthread_mat_vec_struct.c
#include <math.h>
#include <pthread.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

#define MAX_THREADS 64
#define NTIMES 20

// All state a thread needs is packed here, so the thread function reads no
// globals. Each thread receives the address of its own element of an array of
// ThreadData, which keeps every thread's argument object private and alive for
// the whole lifetime of the thread.
typedef struct {
  long my_rank;
  long rows;
  long cols;
  int thread_count;
  const float* A;
  const float* x;
  float* y;
} ThreadData;

static double get_time_ms(void) {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

static const char* check_diff(const float* ref, const float* test, long n) {
  for (long i = 0; i < n; ++i) {
    if (fabs(ref[i] - test[i]) > 1e-5 * fabs(ref[i]) + 1e-5) {
      return "FAIL";
    }
  }
  return "PASS";
}

// Computes rows [first_row, last_row) of y = A*x. The serial baseline and the
// worker threads both call this function, so the two timings compare the same
// machine code instead of two separately compiled copies of the same loop.
static void MatVec_rows(const float* A_in, const float* x_in, float* y_out,
                        long first_row, long last_row, long n_cols) {
  for (long i = first_row; i < last_row; ++i) {
    float sum = 0.0f;
    for (long j = 0; j < n_cols; ++j) {
      sum += A_in[i * n_cols + j] * x_in[j];
    }
    y_out[i] = sum;
  }
}

// Thread entry function. Unpacks its argument struct; touches no globals.
void* MatVec_thread(void* arg) {
  ThreadData* data = (ThreadData*)arg;

  long my_rank = data->my_rank;
  long rows = data->rows;
  long cols = data->cols;
  const float* A = data->A;
  const float* x = data->x;
  float* y = data->y;

  long local_rows = rows / data->thread_count;
  long my_first_row = my_rank * local_rows;
  long my_last_row =
      (my_rank == data->thread_count - 1) ? rows : my_first_row + local_rows;

  // Each thread writes a disjoint slice of y, so no synchronization is needed.
  // Repeat inside the thread so the workers are created once instead of
  // NTIMES times. Short-lived threads are placed poorly by the scheduler on
  // heterogeneous (big.LITTLE) CPUs, which distorts the measurement.
  for (int t = 0; t < NTIMES; ++t) {
    MatVec_rows(A, x, y, my_first_row, my_last_row, cols);
  }

  return NULL;
}

int main(int argc, char* argv[]) {
  if (argc != 4) {
    fprintf(stderr, "Usage: %s <rows> <cols> <thread_count>\n", argv[0]);
    return 1;
  }

  long rows = strtol(argv[1], NULL, 10);
  long cols = strtol(argv[2], NULL, 10);
  int thread_count = (int)strtol(argv[3], NULL, 10);

  if (rows <= 0 || cols <= 0) {
    fprintf(stderr, "Error: rows and cols must be positive\n");
    return 1;
  }
  if (thread_count <= 0 || thread_count > MAX_THREADS) {
    fprintf(stderr, "Error: thread_count must be between 1 and %d\n",
            MAX_THREADS);
    return 1;
  }

  float* A = malloc(rows * cols * sizeof(float));
  float* x = malloc(cols * sizeof(float));
  float* y_parallel = malloc(rows * sizeof(float));
  float* y_serial = malloc(rows * sizeof(float));
  pthread_t* thread_handles = malloc(thread_count * sizeof(pthread_t));
  ThreadData* thread_data = malloc(thread_count * sizeof(ThreadData));

  if (A == NULL || x == NULL || y_parallel == NULL || y_serial == NULL ||
      thread_handles == NULL || thread_data == NULL) {
    fprintf(stderr, "Error: memory allocation failed\n");
    return 1;
  }

  for (long i = 0; i < rows * cols; ++i) A[i] = (float)(i % 100) * 0.01f;
  for (long j = 0; j < cols; ++j) x[j] = (float)(j % 50) * 0.02f;

  printf("Matrix-Vector Multiply (struct arguments)\n");
  printf("Size: %ld x %ld, Threads: %d, Repeats: %d\n\n", rows, cols,
         thread_count, NTIMES);

  double start = get_time_ms();
  for (int t = 0; t < NTIMES; ++t) {
    MatVec_rows(A, x, y_serial, 0, rows, cols);
  }
  double time_serial = (get_time_ms() - start) / NTIMES;

  start = get_time_ms();
  for (long i = 0; i < thread_count; ++i) {
    thread_data[i].my_rank = i;
    thread_data[i].rows = rows;
    thread_data[i].cols = cols;
    thread_data[i].thread_count = thread_count;
    thread_data[i].A = A;
    thread_data[i].x = x;
    thread_data[i].y = y_parallel;

    int rc = pthread_create(&thread_handles[i], NULL, MatVec_thread,
                            (void*)&thread_data[i]);
    if (rc != 0) {
      fprintf(stderr, "Error: pthread_create failed\n");
      return 1;
    }
  }
  for (long i = 0; i < thread_count; ++i) {
    pthread_join(thread_handles[i], NULL);
  }
  double time_parallel = (get_time_ms() - start) / NTIMES;

  printf("Serial time   : %8.3f ms\n", time_serial);
  printf("Parallel time : %8.3f ms\n", time_parallel);
  printf("Speedup       : %8.2fx\n", time_serial / time_parallel);
  printf("Check         : %s\n", check_diff(y_serial, y_parallel, rows));

  free(A);
  free(x);
  free(y_parallel);
  free(y_serial);
  free(thread_handles);
  free(thread_data);
  return 0;
}

编译并运行版本二，参数与版本一完全相同，以便直接比较。

In [ ]:
gemv_struct = compile_c(f"{SRC_DIR}/pthread_mat_vec_struct.c",
                        f"{SRC_DIR}/pthread_mat_vec_struct")
print()
out_struct = run_bin(gemv_struct, 4000, 4000, NT)

## 7. 两个版本的对比

### 7.1 性能对比

两个版本的**计算逻辑完全相同**，差别仅在于线程启动前如何组织参数。传参属于一次性开销，与内层 $O(mn)$ 的计算量相比可以忽略，因此二者性能应当基本一致。

In [ ]:
r_glob = parse_result(out_global)
r_strc = parse_result(out_struct)

print(f"{'版本':<16}{'串行(ms)':>12}{'并行(ms)':>12}{'加速比':>10}{'校验':>8}")
print("-" * 60)
for name, r in [("全局变量传参", r_glob), ("参数结构体传参", r_strc)]:
    print(f"{name:<16}{r['serial']:>12.3f}{r['parallel']:>12.3f}"
          f"{r['speedup']:>9.2f}x{r['check']:>8}")

diff = abs(r_glob["parallel"] - r_strc["parallel"]) / r_glob["parallel"] * 100
print(f"\n两版并行耗时相对差异：{diff:.1f}%")
if diff < 10:
    print("差异在测量噪声范围内，符合预期：传参方式不影响计算性能。")
else:
    print("差异较大，建议重复测量以排除系统负载波动的干扰。")

### 7.2 工程质量对比

性能既然相当，选择的依据就落在工程质量上：

<!--
| 评价维度 | 版本一（全局变量） | 版本二（参数结构体） |
|---|---|---|
| 函数可复用性 | 与特定全局对象绑定，不可复用 | **与数据解耦，可处理任意矩阵** |
| 可重入性 | 不可重入 | **可重入** |
| 命名空间 | 污染全局作用域 | **数据封装于结构体内** |
| 可推理性 | 需通读函数体才知其读写范围 | **从签名与结构体定义即可判断** |
| 只读意图的表达 | 无法表达 | **可用 `const` 在类型层面表达** |
| 代码量 | 较少 | 略多（需定义结构体并逐字段赋值） |
-->
<table>
  <thead>
    <tr>
      <th style="text-align: left;">评价维度</th>
      <th style="text-align: left;">版本一（全局变量）</th>
      <th style="text-align: left;">版本二（参数结构体）</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">函数可复用性</td>
      <td style="text-align: left;">与特定全局对象绑定，不可复用</td>
      <td style="text-align: left;"><strong>与数据解耦，可处理任意矩阵</strong></td>
    </tr>
    <tr>
      <td style="text-align: left;">可重入性</td>
      <td style="text-align: left;">不可重入</td>
      <td style="text-align: left;"><strong>可重入</strong></td>
    </tr>
    <tr>
      <td style="text-align: left;">命名空间</td>
      <td style="text-align: left;">污染全局作用域</td>
      <td style="text-align: left;"><strong>数据封装于结构体内</strong></td>
    </tr>
    <tr>
      <td style="text-align: left;">可推理性</td>
      <td style="text-align: left;">需通读函数体才知其读写范围</td>
      <td style="text-align: left;"><strong>从签名与结构体定义即可判断</strong></td>
    </tr>
    <tr>
      <td style="text-align: left;">只读意图的表达</td>
      <td style="text-align: left;">无法表达</td>
      <td style="text-align: left;"><strong>可用 <code>const</code> 在类型层面表达</strong></td>
    </tr>
    <tr>
      <td style="text-align: left;">代码量</td>
      <td style="text-align: left;">较少</td>
      <td style="text-align: left;">略多（需定义结构体并逐字段赋值）</td>
    </tr>
  </tbody>
</table>

> **结论**：参数结构体的收益体现在**工程质量**上，而非性能上。在超出教学规模的任何程序中，都应当采用参数结构体方案。

## 8. 扩展性：加速比随线程数的变化

下面固定矩阵规模为 $4000 \times 4000$，改变线程数，测量加速比的变化趋势。

In [ ]:
cands = [1, 2, 4, 8]        # 固定测量四个线程数，便于横向比较
ts, sp = [], []
for t in cands:
    r = parse_result(run_bin(gemv_struct, 4000, 4000, t, echo=False))
    ts.append(t)
    sp.append(r["speedup"])
    print(f"{t:2d} 线程 -> 并行耗时 {r['parallel']:8.3f} ms，加速比 {r['speedup']:.2f}x，校验 {r['check']}")

plot_speedup(ts, sp,
             title=f"GEMV 4000x4000 speedup vs thread count "
                   f"({os.cpu_count()} cores)",
             xlabel="Threads")

# 基准自检：1 线程的并行版本与串行版本工作量相同，加速比应接近 1.0
if ts and ts[0] == 1:
    s1 = sp[0]
    if s1 < 0.85 or s1 > 1.15:
        print(f"\n⚠️  1 线程加速比为 {s1:.2f}x，明显偏离 1.0。")
        print("    并行版本在单线程下与串行版本工作量完全相同，二者本应接近。")
        print("    偏离说明基准存在系统性差异，后续加速比数据不可直接采信。")
        print("    常见成因：(a) 串行与并行使用了不同的计算代码；")
        print("              (b) NUMA 首次接触效应，工作线程访问的是远端内存。")
        print("    排查方法见下方「结果解读」。")
    else:
        print(f"\n✅ 基准自检通过：1 线程加速比 {s1:.2f}x，接近 1.0。")

if max(sp) < 1.2:
    print(f"[说明] 本机核心数 = {os.cpu_count()}，各线程数下的加速比均接近 1。")
    print("这一结果本身即为重要结论：线程数超过可用核心数时，线程只能分时复用同一个核心，")
    print("总计算量不变，反而增加了创建与调度开销，因此不会带来任何加速。")
    print("请在鲲鹏多核平台上重跑本单元，以观察真实的加速比曲线。")

### 结果解读

若观察到加速比明显低于线程数，可能的原因有三，需要区分对待：

**① 核心数不足。** 加速比不可能超过物理核心数。请首先核对第 4 节环境检查输出的核心数。

**② 访存受限（memory-bound）。** 这是 GEMV 的**固有特征**，也是最主要的原因。考察内层循环：

```c
sum += A[i * cols + j] * x[j];
```

每完成一次浮点乘加（2 次浮点运算），需要从内存读取一个 `A` 的元素（4 字节）。矩阵 $A$ 的每个元素只被使用一次，没有任何复用机会。其**计算访存比**约为 2 FLOP / 4 Byte = 0.5 FLOP/Byte，远低于现代处理器的平衡点。

这意味着限制性能的是**内存带宽**而非计算能力。多个线程会争抢同一条内存通道，因此加速比通常在核心数远未用满时便趋于平缓。继续增加线程只会加剧带宽争抢。

> 这与第三章的结论完全一致：**优化必须匹配瓶颈**。对访存受限的问题，无论是增加 SIMD 宽度还是增加线程数，收益都会迅速饱和。

**③ 线程创建开销。** 本实验重复 `NTIMES = 20` 次，每次都重新创建并销毁全部线程。当矩阵规模较小时，这部分固定开销的占比会相当显著。实验八将介绍线程池思想以摊薄该开销。

**④ 核心异构与线程放置。** 在**大小核**处理器（ARM big.LITTLE、Intel P-core/E-core）上，这一因素往往最为显著，且极易被误判为并行本身的问题。

Linux 的能效感知调度依据线程累积的负载估计来决定其运行在大核还是小核上。由此产生两个不对称：

- **串行基准**由主线程连续执行 `NTIMES` 次，负载估计持续累积，通常会被迁移到**大核**并保持；
- **工作线程**若在每次迭代都重新创建，则每个新线程的负载历史均为零，会被放置到**小核**上；而它只存活十余毫秒便退出，来不及积累到足以迁移至大核的利用率。

其结果是「串行跑在大核、并行跑在小核」，测得的加速比反映的是核心类型差异，而非并行效果。典型表现为：**1 线程的加速比约为 0.5**，且线程数较少时扩展性异常，直到线程数增多、开始占用大核后才恢复正常。

**本实验的应对措施**：将 `NTIMES` 重复循环置于**线程函数内部**，使工作线程只创建一次、存活足够长的时间，从而获得与主线程相当的调度待遇：

```c
void *MatVec_thread(void *arg) {
  ...
  for (int t = 0; t < NTIMES; ++t) {                 // 重复置于线程内部
    MatVec_rows(A, x, y, my_first_row, my_last_row, cols);
  }
  return NULL;
}
```

这一改动同时消除了 `NTIMES × thread_count` 次线程创建的开销，使测量更准确。

**若仍需进一步排除干扰**，可用 `taskset` 将进程绑定到同一类核心（核心编号见第 4 节的检查结果）：

```bash
taskset -c 0-7 ./src_gemv/02_pthread_mat_vec_struct 4000 4000 4
```

**⑤ NUMA 首次接触效应。** 在多 NUMA 节点的服务器上还需考虑这一因素。Linux 采用首次接触策略分配物理页：页面被分配到**首次写入它的线程所在的节点**。本实验中矩阵 `A` 由主线程初始化，其页面全部位于主线程所在节点；工作线程若被调度到其他节点，访问 `A` 需跨节点互联，带宽显著下降。

用 `lscpu | grep -i numa` 可查看节点数；若为 1 则可排除本因素。根本的解决办法是让**各线程初始化自己后续负责的行区间**（「谁计算，谁初始化」），使数据页从一开始就分布在将要使用它的节点上。这属于 NUMA 感知优化，超出本实验范围，但在多路服务器程序中至关重要。

## 9. 线程创建与销毁的开销

第 8 节的测量把 `NTIMES` 重复循环放在**线程函数内部**，工作线程因而只创建一次。这一安排并非随意为之，本节说明其依据，并定量测出线程创建与销毁的实际代价。

### 9.1 为什么要把重复循环移入线程

一种直觉的写法是把重复循环放在外层，每一轮都重新创建与汇合全部线程：

```c
for (int t = 0; t < NTIMES; ++t) {          // 外层重复
  for (i...) pthread_create(...);           // 每轮都重新创建
  for (i...) pthread_join(...);
}
```

这样写会引入两个独立的问题：

1. **计入了 `NTIMES × thread_count` 次线程创建开销**。测得的便不再是计算部分的并行扩展性，而是「计算 + 反复创建线程」的混合结果。
2. **在大小核处理器上导致线程被放置到小核**。Linux 的能效感知调度依据线程累积的负载估计决定其运行位置。每轮新建的线程负载历史为零，会被放到小核上；而它只存活十余毫秒便退出，来不及积累到足以迁移至大核的利用率。与此同时，连续执行的主线程（串行基准）却会被迁移到大核。二者由此运行在不同类型的核心上，加速比完全失真。

把重复循环移入线程后，工作线程只创建一次、存活足够长的时间，上述两个问题同时消除。

> **这不是回避问题，而是分离变量。** 计算的并行扩展性与线程创建开销是两个独立的性能维度，应当分别测量。下面就单独测量后者。

### 9.2 测量纯粹的创建与销毁开销

要测出线程创建的代价，线程函数必须**不做任何工作**——否则测到的是计算与开销的混合值：

```c
void *Empty_thread(void *arg) {
  (void)arg;          // 显式忽略参数，避免 -Wunused-parameter 警告
  return NULL;
}
```

程序先执行一轮不计时的预热，以排除线程库与内核的一次性初始化成本，随后重复 `rounds` 轮「创建 `thread_count` 个线程并全部汇合」，取平均。

In [ ]:
%%writefile {SRC_DIR}/pthread_create_overhead.c
#include <pthread.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

#define MAX_THREADS 64

// The thread body is empty on purpose: the measured time is then pure creation
// and join overhead, with no useful work mixed in.
void* Empty_thread(void* arg) {
  (void)arg;
  return NULL;
}

static double get_time_ms(void) {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

// Creates thread_count threads, joins them all, and returns 0 on success.
static int spawn_and_join(pthread_t* handles, int thread_count) {
  long created = 0;
  for (long i = 0; i < thread_count; ++i) {
    if (pthread_create(&handles[i], NULL, Empty_thread, NULL) != 0) {
      break;
    }
    ++created;
  }
  for (long i = 0; i < created; ++i) {
    pthread_join(handles[i], NULL);
  }
  return (created == thread_count) ? 0 : 1;
}

int main(int argc, char* argv[]) {
  if (argc != 3) {
    fprintf(stderr, "Usage: %s <thread_count> <rounds>\n", argv[0]);
    return 1;
  }

  int thread_count = (int)strtol(argv[1], NULL, 10);
  long rounds = strtol(argv[2], NULL, 10);

  if (thread_count <= 0 || thread_count > MAX_THREADS) {
    fprintf(stderr, "Error: thread_count must be between 1 and %d\n",
            MAX_THREADS);
    return 1;
  }
  if (rounds <= 0) {
    fprintf(stderr, "Error: rounds must be positive\n");
    return 1;
  }

  pthread_t* thread_handles = malloc(thread_count * sizeof(pthread_t));
  if (thread_handles == NULL) {
    fprintf(stderr, "Error: memory allocation failed\n");
    return 1;
  }

  // One untimed round first, so the measurement does not include one-off
  // library and kernel initialization.
  if (spawn_and_join(thread_handles, thread_count) != 0) {
    fprintf(stderr, "Error: pthread_create failed\n");
    free(thread_handles);
    return 1;
  }

  double start = get_time_ms();
  for (long r = 0; r < rounds; ++r) {
    if (spawn_and_join(thread_handles, thread_count) != 0) {
      fprintf(stderr, "Error: pthread_create failed\n");
      free(thread_handles);
      return 1;
    }
  }
  double elapsed = get_time_ms() - start;

  printf("Thread create/join overhead\n");
  printf("Threads per round: %d, Rounds: %ld\n\n", thread_count, rounds);
  printf("Total time      : %10.3f ms\n", elapsed);
  printf("Time per round  : %10.4f ms\n", elapsed / (double)rounds);
  printf("Cost per thread : %10.2f us\n",
         elapsed * 1000.0 / (double)(rounds * thread_count));

  free(thread_handles);
  return 0;
}

In [ ]:
overhead = compile_c(f"{SRC_DIR}/pthread_create_overhead.c",
                     f"{SRC_DIR}/pthread_create_overhead")
print()

ROUNDS = 200
cost_us = {}
print(f"{'线程数':>8}{'每轮耗时(ms)':>16}{'单个线程开销(us)':>20}")
print("-" * 46)
for t in (1, 2, 4, 8):
    out = run_bin(overhead, t, ROUNDS, echo=False)
    per_round = float(re.search(r"Time per round\s*:\s*([\d.]+)", out).group(1))
    per_thread = float(re.search(r"Cost per thread\s*:\s*([\d.]+)", out).group(1))
    cost_us[t] = per_thread
    print(f"{t:>8}{per_round:>16.4f}{per_thread:>20.2f}")

print(f"\n创建并汇合一个线程的代价约为 {min(cost_us.values()):.0f}–{max(cost_us.values()):.0f} 微秒，")
print("相当于数万条指令：其中包含内核任务结构的分配、栈空间的映射与调度器的介入。")

if cost_us[8] > cost_us[1] * 1.2:
    print("\n本机上单个线程的平均开销随线程数上升。这通常出现在可用核心较少时：")
    print("多个创建请求无法真正并行，反而在调度器与内存分配路径上产生竞争。")
elif cost_us[8] < cost_us[1] * 0.8:
    print("\n本机上单个线程的平均开销随线程数下降：核心数充足时，")
    print("多个创建请求可以在不同核心上重叠进行，从而摊薄了单个线程的平均代价。")
else:
    print("\n本机上单个线程的平均开销基本不随线程数变化。")

### 9.3 开销何时变得不可忽略

绝对数值本身说明不了问题，**关键在于它与并行段计算时间的比值**。下面固定线程数，改变矩阵规模，考察若采用「每轮重新创建线程」的写法，开销将占多大比重。

In [ ]:
sizes = [200, 500, 1000, 2000, 4000]
oh_ms = NT * cost_us[NT] / 1000.0        # 创建 NT 个线程的开销（毫秒）

print(f"线程数 = {NT}，每轮创建开销 ≈ {oh_ms:.3f} ms\n")
print(f"{'矩阵规模':>12}{'单次并行计算(ms)':>20}{'开销占比':>12}{'结论':>10}")
print("-" * 58)
for n in sizes:
    r = parse_result(run_bin(gemv_struct, n, n, NT, echo=False))
    ratio = oh_ms / r["parallel"] * 100
    verdict = "可忽略" if ratio < 5 else ("需注意" if ratio < 50 else "不可接受")
    print(f"{n:>6} x {n:<5}{r['parallel']:>18.3f}{ratio:>11.1f}%{verdict:>10}")

print("\n随着矩阵规模减小，计算时间迅速下降，而线程创建开销基本不变，")
print("因此其占比急剧上升。规模足够小时，并行反而慢于串行。")

### 💡 结论与工程含义

**① 线程不是免费的。** 创建并汇合一个线程的代价在数十微秒量级，相当于数万条指令。这一代价来自内核任务结构的分配、栈空间的映射、以及调度器的介入。

**② 判据是比值而非绝对值。** 当单次并行段的计算时间与线程创建开销处于同一量级时，Fork-Join 模型即不再适用。工程上的经验界限是：**并行段的工作量应当足够大，使线程创建开销占比低于百分之几**。

**③ 反复的 Fork-Join 应当替换为线程池。** 若程序需要多次进入并行段（如本实验的 `NTIMES` 次重复，或服务端逐个处理请求），正确的做法是**创建一次线程，令其反复取用任务**，而非每次都创建与销毁。这就是**线程池**（thread pool）的基本思想。

实现线程池需要解决一个新问题：工作线程完成一轮后必须**等待**下一批任务到来，而不能空转或退出。这需要线程之间的**事件通知**机制——实验四将引入信号量，实验五将用它构建生产者—消费者模型，届时即可给出完整的线程池实现。

> 本实验把重复循环移入线程，实际上就是线程池思想最简单的一种特例：线程创建一次，完成全部 `NTIMES` 轮工作后才退出。它之所以可行，是因为各轮任务在编译期即已确定、无需运行时分派。

## 10. 结果分析

本实验建立了三项贯穿全章的认识：

**① 并非所有共享数据都需要同步。** 判据是「是否存在对同一内存位置的并发访问，且其中至少有一个是写操作」，而非「数据是否共享」。只读共享与写入互不重叠的区间，均无需任何同步机制。

**② 传参方式的选择依据是工程质量，不是性能。** 两个版本性能相当，但参数结构体在可复用性、可重入性、可推理性上全面占优。

**③ 加速比受制于瓶颈类型。** GEMV 属访存受限问题，其加速比曲线会在核心数用满之前趋于平缓。判断优化方向之前，必须先识别瓶颈所在。

### 🎓 结论

本实验的并行实现**没有使用任何锁，却是完全正确的**——这一点极为重要。学习并发编程时容易形成一种倾向：只要涉及多线程就加锁。本实验表明，正确的做法是先分析数据的访问模式，仅在确有并发写冲突时才引入同步机制。

从实验三开始，我们将处理确实存在写冲突的情形。届时可以对照本实验，体会两类问题在本质上的差异。

## 11. 🔧 动手练习

请修改代码、重新编译并运行，观察行为与性能的变化：

1. 将行块划分改为**循环划分**（线程 $r$ 负责第 $r,\ r+t,\ r+2t,\ \dots$ 行），测量性能变化，并结合缓存局部性解释所得结果。
2. 构造负载不均衡的场景：将矩阵改为下三角形式（仅计算 $j \le i$ 的部分），观察块划分下的加速比，说明其为何劣化，并给出改进的划分方案。
3. 将版本二中的 `const float *A` 改为 `float *A`，在线程函数中尝试写入 `A`，观察编译器是否报错，说明 `const` 在并发代码中的价值。
4. 故意改用单个 `ThreadData data;` 传给所有线程，在线程函数中打印各自的 `my_rank`，运行多次并记录错误现象，解释其成因。
5. 将 `NTIMES` 由 20 改为 1，对比小矩阵（如 $100 \times 100$）下两个版本的耗时，据此估算线程创建与销毁的固定开销。

## 12. 🤔 思考题

- 本实验的并行版本没有使用任何锁，为何仍然正确？请用数据竞争的严格定义逐条论证。
- 若将划分方式改为「线程 0 负责所有偶数行、线程 1 负责所有奇数行」，是否仍然不需要锁？这样做会引入什么新的性能问题？（提示：参见实验八。）
- `pthread_create` 的第四个参数类型为 `void *`。标准为何不将其设计为可变参数（如同 `printf`），从而直接传递多个参数？
- 假设矩阵大到无法一次装入内存，需要分块从磁盘读取。此时「每个线程写不重叠区间」的结论是否依然成立？还需要额外考虑什么？
- 本实验测得的加速比通常明显低于线程数。请设计一个计算密集型的替代任务（提高计算访存比），并预测其加速比曲线与本实验有何不同。

## 13. 小结与后续

本实验通过同一算法的两种传参实现，完成了从「传递一个整数」到「传递完整数据集」的过渡：

<!--
| 版本 | 传参方式 | 新增知识点 |
|---|---|---|
| **版本一** | 全局变量 | 按行块划分、无锁并行的判据、平均值计时法 |
| **版本二** | 参数结构体 | 每线程独立参数对象、参数生命周期、`const` 表达只读意图 |
| **开销测量** | 空线程体 | 线程创建代价的量级、Fork-Join 的适用边界 |
-->
<table>
  <thead>
    <tr>
      <th style="text-align: left;">版本</th>
      <th style="text-align: left;">传参方式</th>
      <th style="text-align: left;">新增知识点</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><strong>版本一</strong></td>
      <td style="text-align: left;">全局变量</td>
      <td style="text-align: left;">按行块划分、无锁并行的判据、平均值计时法</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>版本二</strong></td>
      <td style="text-align: left;">参数结构体</td>
      <td style="text-align: left;">每线程独立参数对象、参数生命周期、<code>const</code> 表达只读意图</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>开销测量</strong></td>
      <td style="text-align: left;">空线程体</td>
      <td style="text-align: left;">线程创建代价的量级、Fork-Join 的适用边界</td>
    </tr>
  </tbody>
</table>

两版性能相当，但版本二在工程质量上全面占优，应作为此后所有实验的默认写法。

本实验还给出了性能测量的三项方法学要求，它们在此后各实验中同样适用：

1. **被比较的版本必须执行同一段计算代码**（第 5 节），否则测得的是编译器差异；
2. **各次测量必须运行在同类型的核心上**（第 4 节），否则测得的是大小核差异；
3. **计算的扩展性与线程创建开销应分别测量**（第 9 节），否则两者互相污染。

其中第 1、3 项可用同一个判据自检：**将线程数设为 1，加速比应接近 1.0**。

➡️ **后续内容：实验三 π 估算：数据竞争与锁粒度**。本实验的并行之所以无需同步，前提是各线程写入互不重叠的区间。当这一前提被打破——多个线程需要累加到**同一个**变量时，将出现本章第一个真正的并发缺陷：**数据竞争**。届时程序会给出错误的计算结果，且每次运行的结果都不相同。